In [ ]:
# 핵심: pandas에서 정제한 표 데이터를 PyTorch가 계산할 수 있는 tensor 형식으로 옮기는 단계다.
from pathlib import Path

import pandas as pd
import torch

data_file = Path("data/house_tiny.csv")

if not data_file.exists():
    raise FileNotFoundError("먼저 01_reading_the_dataset.ipynb를 실행하세요.")

data = pd.read_csv(data_file)

print("type(data):", type(data))
print(data)

type(data): <class 'pandas.DataFrame'>
   NumRooms RoofType   Price
0       NaN      NaN  127500
1       2.0      NaN  106000
2       4.0    Slate  178100
3       NaN      NaN  140000


In [9]:
# 입력 특징과 예측 목표를 분리한다.
# 핵심: 범주형 열을 one-hot encoding하고 수치형 결측값을 채운 뒤에야 직사각형 수치 배열로 변환할 수 있다.
from unicodedata import numeric


inputs = data.iloc[:, 0:2].copy()
targets = data.iloc[:, 2].copy()

# 범주형 데이터를 dummy 변수로 변환한다.
inputs = pd.get_dummies(inputs, dummy_na=True)

# 수치형 열의 결측값을 평균으로 채운다.
numeric_columns = inputs.select_dtypes(include="number").columns

inputs[numeric_columns] = inputs[numeric_columns].fillna(
    inputs[numeric_columns].mean()
)

print("type(data):", type(data))
print(data)

print("\nPrepared inputs:")
print(inputs)

print("\nTargets:")
print(targets)

type(data): <class 'pandas.DataFrame'>
   NumRooms RoofType   Price
0       NaN      NaN  127500
1       2.0      NaN  106000
2       4.0    Slate  178100
3       NaN      NaN  140000

Prepared inputs:
   NumRooms  RoofType_Slate  RoofType_nan
0       3.0           False          True
1       2.0           False          True
2       4.0            True         False
3       3.0           False          True

Targets:
0    127500
1    106000
2    178100
3    140000
Name: Price, dtype: int64


In [10]:
# pandas 데이터를 float32 NumPy 배열로 먼저 변환한다.
# 핵심: 딥러닝 입력은 보통 float32를 사용하며 from_numpy는 NumPy 배열과 CPU 메모리를 공유한다.
X_numpy = inputs.to_numpy(dtype="float32")
y_numpy = targets.to_numpy(dtype="float32")

# from_numpy는 numpy 배열과 메모리를 공유하는 텐서를 만든다.
X = torch.from_numpy(X_numpy)
y = torch.from_numpy(y_numpy)

print("Input tensor X:")
print(X)

print("\nTarget tensor y:")
print(y)

Input tensor X:
tensor([[3., 0., 1.],
        [2., 0., 1.],
        [4., 1., 0.],
        [3., 0., 1.]])

Target tensor y:
tensor([127500., 106000., 178100., 140000.])


In [12]:
# 핵심: X의 shape는 (사례 수, 특징 수), y의 shape는 (사례 수,)이고 입력 데이터는 보통 gradient를 요구하지 않는다.
print("X shape:", X.shape)
print("X dtype:", X.dtype)

print("\ny shape:", y.shape)
print("y dtype:", y.dtype)

print("\nX requires_grad:", X.requires_grad)
print("y requires_grad:", y.requires_grad)

assert X.shape == (4, 3)
assert y.shape == (4,)
assert X.dtype == torch.float32
assert y.dtype == torch.float32
assert not torch.isnan(X).any()
assert not torch.isnan(y).any()

X shape: torch.Size([4, 3])
X dtype: torch.float32

y shape: torch.Size([4])
y dtype: torch.float32

X requires_grad: False
y requires_grad: False
